In [7]:
import pandas as pd
from google.colab import files

uploaded = files.upload()
df = pd.read_csv('Jemilah_Youtube_Review_Transcript.csv')

Saving Jemilah_Youtube_Review_Transcript.csv to Jemilah_Youtube_Review_Transcript.csv


In [9]:
import nltk
import string
from nltk.corpus import stopwords
nltk.download('stopwords')

def clean_text(text):
    if pd.isnull(text):
        return ""
    text = text.replace('\u2019', "'")
    text = text.replace('&amp;', 'and')
    tokens = text.lower().translate(str.maketrans('', '', string.punctuation)).split()
    return ' '.join([word for word in tokens if word not in stopwords.words('english')])

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [10]:
df['transcript_cleaned'] = df['Transcript'].apply(clean_text)
print(df[['Transcript', 'transcript_cleaned']].head())

                                          Transcript  \
0  My base pay is $18.50 an hour, and my overtime...   
1  I’ve been doing flex, and I decided to try ove...   
2  An Amazon worker died on the job in a warehous...   
3  Most of the work at Amazon is not difficult. I...   
4  It is more important to organize than ever bef...   

                                  transcript_cleaned  
0  base pay 1850 hour overtime pay 2750 hour work...  
1  ive flex decided try overtime maybe twice thre...  
2  amazon worker died job warehouse troutdale ore...  
3  work amazon difficult boring repetitive theres...  
4  important organize ever today reminded importa...  


One limitation I noticed was that “$18.50” and “$27.50” lost their dollar signs and decimal points during cleaning, becoming “1850” and “2750.” This happened because the dollar sign and period were removed as punctuation, causing the numbers to merge. Since this cleaning step is intended for general sentiment and topic analysis rather than precise financial analysis, I left it as a noted limitation. The original transcript column still preserves the exact pay amounts for reference.

In [11]:
print(df['transcript_cleaned'].head()) ## checklist 1

0    base pay 1850 hour overtime pay 2750 hour work...
1    ive flex decided try overtime maybe twice thre...
2    amazon worker died job warehouse troutdale ore...
3    work amazon difficult boring repetitive theres...
4    important organize ever today reminded importa...
Name: transcript_cleaned, dtype: object


In [12]:
print(df['transcript_cleaned'].str.contains(r'\bthe\b|\bjust\b|\band\b|\bis\b|\bwas\b', regex=True).sum()) ## checklist 2

1


In [13]:
print(df[['Transcript', 'transcript_cleaned']].head(10)) ## checklist 3

                                          Transcript  \
0  My base pay is $18.50 an hour, and my overtime...   
1  I’ve been doing flex, and I decided to try ove...   
2  An Amazon worker died on the job in a warehous...   
3  Most of the work at Amazon is not difficult. I...   
4  It is more important to organize than ever bef...   
5  It’s an hour bus ride, by the way. The bus rid...   
6  It’s a pretty boring job, but it’s one of the ...   
7  After this 10-hour shift, I couldn’t feel my l...   
8  Working there taught me a great lesson, it rea...   
9  I’m working inbound, so that’s basically recei...   

                                  transcript_cleaned  
0  base pay 1850 hour overtime pay 2750 hour work...  
1  ive flex decided try overtime maybe twice thre...  
2  amazon worker died job warehouse troutdale ore...  
3  work amazon difficult boring repetitive theres...  
4  important organize ever today reminded importa...  
5  hour bus ride way bus ride hour 15 30 minutes ... 

I investigated further and found that the issue was a left curly quote (Unicode U+2018) before “Just.” It was similar to the right curly quote issue I found earlier, but my existing fix only handled the right curly quote (U+2019), so the left curly quote was still being missed.

In [14]:
mask = df['transcript_cleaned'].str.contains(r'\bthe\b|\bjust\b|\band\b|\bis\b|\bwas\b', regex=True)
print(df[mask][['Transcript', 'transcript_cleaned']])

                                          Transcript  \
2  An Amazon worker died on the job in a warehous...   

                                  transcript_cleaned  
2  amazon worker died job warehouse troutdale ore...  


In [16]:
print(repr(df['Transcript'][2]))

"An Amazon worker died on the job in a warehouse in Troutdale, Oregon. As he lay dead on the floor for over an hour, workers were instructed to continue doing their jobs around his body. A worker tried to help him, and a manager said, ‘Just turn around and not look'. Let’s get back to work"


In [17]:
test = "manager said ‘Just turn around"
print(clean_text(test))

manager said ‘just turn around


In [18]:
def clean_text(text):
    if pd.isnull(text):
        return ""
    text = text.replace('\u2019', "'")
    text = text.replace('\u2018', "'")
    text = text.replace('&amp;', 'and')
    tokens = text.lower().translate(str.maketrans('', '', string.punctuation)).split()
    return ' '.join([word for word in tokens if word not in stopwords.words('english')])

In [24]:
test = "manager said ‘Just turn around"
print(clean_text(test))

manager said turn around


In [21]:
df['transcript_cleaned'] = df['Transcript'].apply(clean_text)
print(df[['Transcript', 'transcript_cleaned']].head(10))

                                          Transcript  \
0  My base pay is $18.50 an hour, and my overtime...   
1  I’ve been doing flex, and I decided to try ove...   
2  An Amazon worker died on the job in a warehous...   
3  Most of the work at Amazon is not difficult. I...   
4  It is more important to organize than ever bef...   
5  It’s an hour bus ride, by the way. The bus rid...   
6  It’s a pretty boring job, but it’s one of the ...   
7  After this 10-hour shift, I couldn’t feel my l...   
8  Working there taught me a great lesson, it rea...   
9  I’m working inbound, so that’s basically recei...   

                                  transcript_cleaned  
0  base pay 1850 hour overtime pay 2750 hour work...  
1  ive flex decided try overtime maybe twice thre...  
2  amazon worker died job warehouse troutdale ore...  
3  work amazon difficult boring repetitive theres...  
4  important organize ever today reminded importa...  
5  hour bus ride way bus ride hour 15 30 minutes ... 

In [22]:
print(df['transcript_cleaned'].str.contains(r'\bthe\b|\bjust\b|\band\b|\bis\b|\bwas\b', regex=True).sum()) ## checklist 2 showing 0

0


In [25]:
df.to_csv('youtube_cleaned_text.csv', index=False)